In [7]:
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

from process_data import pre_process_data


con = duckdb.connect('../capillary.db')

df = con.execute("""
                SELECT id,
                    any_value(pid) as pid, 
                    any_value(value) as value,
                    any_value(analysis) as analysis,
                    any_value(protein_value) as protein_value,
                    list(c.comment_nr) as comment_nr,
                    any_value(sign) as sign,
                    any_value(age) as age,
                FROM protein_data p
                LEFT JOIN comment_precences c 
                    ON p.id = c.row_id
                GROUP BY id
                 """).df()

pids = con.execute("SELECT DISTINCT pid FROM protein_data").df()

con.close()

def to_severity(comments: list[int]):
    max_found = 0
    for comment in comments:
        result = map_severity(comment)
        if result > max_found:
            max_found = result
    return max_found
        

def map_severity(label: int) -> int:        
    severity = 0
    match label:
        case 40 |41 |42:
            severity = 1 #absolut 
        case 43 | 44 | 45:
            severity = 2 #relativ
        case _:
            severity = 0
    return severity



df['severity'] = df['comment_nr'].apply(to_severity)
df['relativ'] = (df['severity'] == 2).astype(int)
df = df[df['age']> 13]

df

,id,pid,value,analysis,protein_value,comment_nr,sign,age,severity,relativ
0,177848,68102067,"[0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 3, ...","[0054, 0056, 0055, 0058, 0062, 0064, 0065, 006...","[43.0, 1.2, 0.78, 1.3, 1.4, 14.9, 3.02, 1.05]","[20, 70, 97]",nxu,34.166667,0,0
1,177852,27601379,"[1, 1, 1, 1, 1, 1, 2, 2, 2, 3, 3, 4, 4, 4, 4, ...","[0054, 0056, 0055, 0058, 0062, 0064, 0065, 006...","[36.0, 1.34, 0.92, 1.32, 5.3, 10.3, 4.8, 1.12]","[70, 17, 97]",nxu,79.916667,0,0
2,177865,15564017,"[0, 0, 1, 1, 1, 1, 1, 2, 3, 3, 3, 3, 2, 2, 2, ...","[0054, 0056, 0055, 0058, 0062, 0064, 0065, 006...","[49.0, 1.31, 0.83, 2.34, 1.3, 15.4, 1.97, 0.87]","[22, 70, 832, 805, 97]",mla,42.416667,0,0
3,177871,79480694,"[3, 2, 2, 2, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 5, ...","[0054, 0056, 0055, 0058, 0062, 0064, 0065, 006...","[47.0, 1.46, 0.65, 1.38, 0.99, 11.7, 2.43, 0.74]","[20, 66, 69, 4, 68, 97]",mla,60.833333,0,0
4,177875,54989220,"[0, 1, 1, 1, 1, 1, 1, 2, 2, 2, 3, 3, 3, 3, 3, ...","[0054, 0056, 0055, 0058, 0062, 0064, 0065, 006...","[40.0, 1.43, 0.9, 1.09, 0.97, 8.27, 1.24, 2.11]","[20, 70, 97]",nxu,75.750000,0,0
...,...,...,...,...,...,...,...,...,...,...
172329,140197,18834247,"[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 2, ...","[0054, 0056, 0055, 0058, 0062, 0064, 0065, 006...","[44.0, 2.04, 1.33, 1.57, 145.0, 11.1, 1.69, 1.01]",[--],mjo,44.166667,0,0
172330,50471,49265653,"[0, 1, 2, 2, 2, 2, 2, 3, 3, 4, 4, 5, 5, 5, 7, ...","[0054, 0056, 0055, 0058, 0062, 0064, 0065, 006...","[43.0, 1.43, 0.86, 1.11, 1.4, 10.2, 2.54, 0.18]",[--],nxu,46.416667,0,0
172331,43979,75987653,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 2, 3, 3, 3, 4, ...","[0054, 0056, 0055, 0058, 0062, 0064, 0065, 006...","[40.0, 1.56, 1.27, 1.53, 9.3, 9.91, 0.66, 0.16]",[--],mjo,66.916667,0,0
172332,47681,77966063,"[2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 2, 2, 2, 1, 0, ...","[0054, 0056, 0055, 0058, 0062, 0064, 0065, 006...","[44.0, 1.4, 0.88, 1.07, 0.52, 7.62, 1.11, 0.19]",[--],lwa,77.250000,0,0


In [8]:
severity_range = [0,1,2]

most_frequent = 0
position_weights = []

for i in severity_range:
    no_occurences = (df['severity'] == i).sum()
    print(f"Antal fall med klass {i}:{no_occurences}")
    if no_occurences > most_frequent:
        most_frequent=no_occurences

for i in severity_range:
    no_occurences = (df['severity'] == i).sum()
    position_weights.append( most_frequent/no_occurences) 


Antal fall med klass 0:166214
Antal fall med klass 1:3460
Antal fall med klass 2:903


In [9]:
valtrain_pids, test_pids = train_test_split(pids, test_size=0.15,random_state=25)
train_pids, val_pids = train_test_split(valtrain_pids, test_size=0.15,random_state=25)

train_set = set(train_pids['pid'])
val_set = set(val_pids['pid'])
print(len(train_set & val_set))  # ska vara 0


test_rows = df[ df['pid'].isin(test_pids['pid'].tolist()) ]
val_rows = df[ df['pid'].isin(val_pids['pid'].tolist()) ]
train_rows = df[ df['pid'].isin(train_pids['pid'].tolist()) ]

print(f"Antal fall relativ stegring i träningsdatan:{(train_rows['severity'] == 2).sum()}")
print(f"Antal fall utan stegring i träningsdatan:{(train_rows['severity'] == 0).sum()}")
print(f"Antal fall med relativ stegring i testdatan:{(test_rows['severity'] == 2).sum()}")
print(f"Antal fall utan stegring i testdatan:{(test_rows['severity'] == 0).sum()}")


0
Antal fall relativ stegring i träningsdatan:625
Antal fall utan stegring i träningsdatan:118767
Antal fall med relativ stegring i testdatan:142
Antal fall utan stegring i testdatan:25397


In [10]:
plt.figure()
plt.hist(train_rows[train_rows['severity'] == 1]['haptoglobin'],bins=100)

KeyError: 'haptoglobin'

<Figure size 640x480 with 0 Axes>

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_recall_curve
import joblib


X = np.vstack(train_rows['protein_value'].to_numpy())
y = np.array(train_rows['relativ'])
X = X[:,0:5]
scaler = StandardScaler()
X = scaler.fit_transform(X)

model = LogisticRegression(class_weight='balanced').fit(X, y)
joblib.dump(model,'haptoglobin.pkl')
joblib.dump(scaler, 'haptoglobin_scaler.pkl')

X_test = np.vstack(test_rows['protein_value'].to_numpy())
X_test = X_test[:,0:5]
y_test = np.array(test_rows['relativ'])
X_test = scaler.transform(X_test)

predictions = model.predict(X_test)
print(classification_report(y_test, predictions, target_names=['Negativ', 'Positiv']))

print(confusion_matrix(y_test, predictions))

probs = model.predict_proba(X_test)[:, 1]


print(model.coef_)
PROTEIN_LABELS = ['Albumin','Orosomukoid','Antitrypsin','Haptoglobin','CRP']
print(PROTEIN_LABELS)

              precision    recall  f1-score   support

     Negativ       1.00      0.92      0.96     25275
     Positiv       0.06      0.96      0.11       134

    accuracy                           0.92     25409
   macro avg       0.53      0.94      0.53     25409
weighted avg       0.99      0.92      0.95     25409

[[23217  2058]
 [    6   128]]
[[-0.52038885  1.2343777  -0.19774563 -5.42052139  1.10074947]]
['Albumin', 'Orosomukoid', 'Antitrypsin', 'Haptoglobin', 'CRP']


## Resultat: 

Över 13 år: alla under 0,24 ska få kommentar 42